# Geomorphic Flood Index (GFI) v2.0 — Google Colab

**Original MATLAB:** Saavedra Navarro et al. | DOI: [10.5281/zenodo.18903835](https://doi.org/10.5281/zenodo.18903835)  
**Python port:** modul `gfi2`

---
### Cara pakai
1. Jalankan **Sel 1** untuk install dependencies
2. Jalankan **Sel 2** untuk clone repo dari GitHub
3. Jalankan **Sel 3** untuk upload file input
4. Pilih **Sel 4A** (MODE A) atau **Sel 4B** (MODE B) sesuai data yang dimiliki
5. Jalankan **Sel 5** untuk download output

In [ ]:
# ── SEL 1: Install dependencies ──────────────────────────────────────────
# Jalankan sekali per sesi Colab
!pip install -q rasterio pysheds richdem numba scipy matplotlib joblib

In [ ]:
# ── SEL 2: Clone repo dari GitHub ────────────────────────────────────────
# Ganti URL dengan repo GitHub Arwan setelah di-push
!git clone https://github.com/YOUR_USERNAME/gfi2-python.git
%cd gfi2-python

import sys
sys.path.insert(0, '.')

from gfi2 import run_gfi2
print('gfi2 berhasil diimport.')

In [ ]:
# ── SEL 3: Upload file input ──────────────────────────────────────────────
from google.colab import files
import os

print('Upload file input (pilih semua sekaligus):')
print('  WAJIB  : DEM mentah (.tif)')
print('  WAJIB  : Flood reference binary (.tif)')
print('  WAJIB  : Simulated water depth (.tif)')
print('  MODE B : DEM filled, Flow Dir ESRI D8, Flow Acc (.tif)')
print()

uploaded = files.upload()
print('\nFile terupload:')
for fname, data in uploaded.items():
    print(f'  {fname}  ({len(data)/1e6:.1f} MB)')

In [ ]:
# ── SEL 4A: Jalankan pipeline — MODE A (hanya DEM mentah) ─────────────────
# Gunakan jika belum punya flow direction / accumulation

import os
os.makedirs('output', exist_ok=True)

results = run_gfi2(
    input_mode         = 'auto',

    # ── Sesuaikan nama file dengan yang diupload ──
    dem_path           = 'Bradano.tif',
    flood_ref_path     = 'Flood T = 200.tif',
    flood_depth_path   = 'Tiranti T = 200.tif',

    # ── Parameter (sesuaikan dengan DAS Anda) ─────
    channel_threshold  = 1e5,          # turunkan untuk DAS kecil
    n_exponent         = 0.354429752,
    roc_step           = 0.005,
    max_iter           = 6,
    n_jobs             = -1,

    out_dir            = 'output',
    save_rasters       = True,
    show_plots         = True,
)

In [ ]:
# ── SEL 4B: Jalankan pipeline — MODE B (raster sudah tersedia) ────────────
# Gunakan jika sudah punya flow dir & acc dari QGIS / ArcGIS / WBT
# CATATAN: flow direction HARUS encoding ESRI D8 (1,2,4,8,16,32,64,128)

import os
os.makedirs('output', exist_ok=True)

results = run_gfi2(
    input_mode         = 'manual',

    # ── Sesuaikan nama file dengan yang diupload ──
    dem_path           = 'DEM.tif',
    demcon_path        = 'DEM_filled.tif',
    flowdir_path       = 'FlowDir_ESRI_D8.tif',
    flowacc_path       = 'FlowAcc.tif',
    flood_ref_path     = 'Flood T = 200.tif',
    flood_depth_path   = 'Tiranti T = 200.tif',

    # ── Parameter ─────────────────────────────────
    channel_threshold  = 1e5,
    n_exponent         = 0.354429752,
    roc_step           = 0.005,
    max_iter           = 6,
    n_jobs             = -1,

    out_dir            = 'output',
    save_rasters       = True,
    show_plots         = True,
)

In [ ]:
# ── SEL 5: Lihat ringkasan hasil ──────────────────────────────────────────
print('=== RINGKASAN HASIL ===')
print(f"AUC  v1.0 : {results['params_v1']['auc']:.4f}")
print(f"AUC  v2.0 : {results['params_v2']['auc']:.4f}")
print(f"RMSE v1.0 : {results['metrics_v1']['rmse']:.4f} m")
print(f"RMSE v2.0 : {results['metrics_v2']['rmse']:.4f} m")
print(f"KGE  v1.0 : {results['metrics_v1']['kge']:.4f}")
print(f"KGE  v2.0 : {results['metrics_v2']['kge']:.4f}")

import os
print('\nFile output:')
for f in os.listdir('output'):
    path = os.path.join('output', f)
    print(f'  {f}  ({os.path.getsize(path)/1e6:.1f} MB)')

In [ ]:
# ── SEL 6: Download semua output ──────────────────────────────────────────
from google.colab import files
import os

for fname in os.listdir('output'):
    fpath = os.path.join('output', fname)
    print(f'Mendownload: {fname}')
    files.download(fpath)

In [ ]:
# ── SEL OPSIONAL: Konversi TauDEM → ESRI (jika flow dir dari TauDEM) ──────
from gfi2 import load_tif, save_tif, convert_taudem_to_esri

taudem_arr, profile = load_tif('FlowDir_TauDEM.tif')
esri_arr = convert_taudem_to_esri(taudem_arr)
save_tif(esri_arr, profile, 'FlowDir_ESRI_D8.tif')
print('Konversi selesai. Gunakan FlowDir_ESRI_D8.tif untuk MODE B.')